In [4]:

from pathlib import Path

sampling_only = False
ensure_proper_read = True
interesting_keys = None
configuration_dict_full = {
    'synflow_64': {
        'seh': {
            ('rgfn_old_r_2', 'gmum', 'RGFN_old_develop'): [
                'rgfn_seh_64_old_code_grid_1_less_reactions_0/2025-04-10_06-32-44/params_0',
                'rgfn_seh_64_old_code_grid_1_less_reactions_1/2025-04-10_17-15-53/params_0',
                'rgfn_seh_64_old_code_grid_1_less_reactions_2/2025-04-10_18-39-56/params_0'
            ],
            ('rgfn_old_r_3', 'gmum', 'RGFN_old_develop'): [
                'rgfn_seh_64_old_code_grid_1_less_reactions_3/2025-04-10_22-03-32/params_0',
                'rgfn_seh_64_old_code_grid_1_less_reactions_4/2025-04-11_04-41-37/params_0',
                'rgfn_seh_64_old_code_grid_1_less_reactions_5/2025-04-12_02-19-55/params_0'
            ],
            (f'synflownet_r_2', 'gmum', 'synflownet'): [
                f'setup_synflow_64_task_seh_reaction_pb_REINFORCE_action_embedding_True_max_len_2_r_1.0_temp_32_seed_{seed}'
                for seed in range(3)
            ],
            (f'synflownet_r_3', 'gmum', 'synflownet'): [
                f'setup_synflow_64_task_seh_reaction_pb_REINFORCE_action_embedding_True_max_len_3_r_1.0_temp_32_seed_{seed}'
                for seed in range(3)
            ],
        }
    }
}



In [5]:
import pandas as pd
import subprocess
from tqdm import tqdm
import os
import tailer
import pandas as pd
import io


def check_path(local_path: Path, configuration, remove_invalid_files=False) -> bool:
    if local_path.exists():
        # if number of lines is lower than 1000, we assume that the file is invalid
        try:
            with open(local_path) as file:
                last_lines = tailer.tail(file, 105)

            df = pd.read_csv(io.StringIO('\n'.join(last_lines)), header=None)
            if len(df) < 100:
                # if remove_invalid_files:
                #     os.remove(local_path)
                return False
        except KeyboardInterrupt:
            raise KeyboardInterrupt
        except Exception as e:
            print(f'Invalid file: {local_path}, {configuration}')
            # if remove_invalid_files:
            #     os.remove(local_path)
            return False
    return True


def analyze_configuration_dict(remove_invalid_files=False):
    for setup_name, task_dict in configuration_dict_full.items():
        for task_name, run_dict in task_dict.items():
            base_dir = Path('results') / setup_name / task_name
            for configuration, paths in run_dict.items():
                local_dir_name, server, server_project_dir = configuration
                for i, path in enumerate(paths):
                    local_training_path = base_dir / local_dir_name / 'training' / f"paths_{i}.csv"
                    local_sampling_path = base_dir / local_dir_name / 'sampling' / f"paths_{i}.csv"
                    # check_path(local_training_path, path, remove_invalid_files)
                    if not check_path(local_sampling_path, path, remove_invalid_files):
                        print(path)


analyze_configuration_dict(remove_invalid_files=False)

In [6]:
def download(input_path, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not output_path.exists():
        cmd = f'scp {input_path} {output_path}'
        if os.system(cmd) == 256 and os.path.exists(output_path):
            os.remove(output_path)
            # raise KeyboardInterrupt
        # if os.system(cmd) == 256:
        #     os.remove(output_path)
        #     raise KeyboardInterrupt
    # elif ensure_proper_read:
    #     # don't print the output of os.system
    #     if os.system(f'tail -n 1 {output_path} > /dev/null 2>&1') != 0:
    #         os.remove(output_path)
    #         download(input_path, output_path)


for templates_name, configuration_dict_template in tqdm(configuration_dict_full.items()):
    # if templates_name not in ['rgfn_new_filtered', 'synflow_64', 'synflow_128']:
    #     continue
    for task_name, configuration_dict in tqdm(configuration_dict_template.items()):
        print(f'{templates_name} {task_name}')
        base_dir = Path('results') / templates_name / task_name
        for configuration, paths in configuration_dict.items():
            dir_name, server, project_dir = configuration
            if interesting_keys and dir_name not in interesting_keys:
                continue
            dir_path = base_dir / dir_name
            dir_path.mkdir(parents=True, exist_ok=True)
            if 'RGFN' in project_dir:
                experiments_dir = 'experiments'
            elif 'guacamol' in project_dir:
                experiments_dir = None
            else:
                experiments_dir = 'logs'
            for i, path in enumerate(paths):
                if experiments_dir:
                    path = Path(project_dir) / experiments_dir / path
                else:
                    path = Path(project_dir) / path
                if not sampling_only:
                    download(f"{server}:{path / 'paths.csv'}", dir_path / 'training' / f"paths_{i}.csv")
                    download(f"{server}:{path / 'molecules.csv'}", dir_path / 'training' / f"molecules_{i}.csv")
                download(f"{server}:{path / 'final_paths.csv'}", dir_path / 'sampling' / f"paths_{i}.csv")
                download(f"{server}:{path / 'final_molecules.csv'}", dir_path / 'sampling' / f"molecules_{i}.csv")


  0%|          | 0/1 [00:00<?, ?it/s]

synflow_64 seh


scp: RGFN_old_develop/experiments/rgfn_seh_64_old_code_grid_1_less_reactions_0/2025-04-10_06-32-44/params_0/molecules.csv: No such file or directory
scp: RGFN_old_develop/experiments/rgfn_seh_64_old_code_grid_1_less_reactions_0/2025-04-10_06-32-44/params_0/final_molecules.csv: No such file or directory
scp: RGFN_old_develop/experiments/rgfn_seh_64_old_code_grid_1_less_reactions_1/2025-04-10_17-15-53/params_0/molecules.csv: No such file or directory
scp: RGFN_old_develop/experiments/rgfn_seh_64_old_code_grid_1_less_reactions_1/2025-04-10_17-15-53/params_0/final_molecules.csv: No such file or directory
scp: RGFN_old_develop/experiments/rgfn_seh_64_old_code_grid_1_less_reactions_2/2025-04-10_18-39-56/params_0/molecules.csv: No such file or directory
scp: RGFN_old_develop/experiments/rgfn_seh_64_old_code_grid_1_less_reactions_2/2025-04-10_18-39-56/params_0/final_molecules.csv: No such file or directory
scp: RGFN_old_develop/experiments/rgfn_seh_64_old_code_grid_1_less_reactions_3/2025-04-1